### Imports

In [1]:
import jax.numpy as jnp
import time
from dynamaxsys.base import get_discrete_time_dynamics
from dynamaxsys.parafoil import (
    JannParafoil4DOF,
    JannParafoil4DOF2,
    JannParafoil3DOF,
    SlegersParafoil6DOF,
)
from model_parameters import (
    slegers_6dof_nonlinear_params,
    jann_4dof_params,
    jann_3dof_params,
)
import controllers

from simulator import (
    simulate,
    simulate_with_controller,
    DEG_TO_RAD,
    RAD_TO_DEG,
    plot_states,
    plot_jann_body,
    plot_3D_traj,
)

### Simulation Hyperparameters

In [2]:
dt = 0.01  # time step size (seconds)
time_horizon = 100  # total time (seconds)
N = int(time_horizon / dt)  # number of timesteps
ts = jnp.arange(0, time_horizon, dt)


# you can observe the singularity when
# time_horizion = 90,
# slegers_initial_state = {
#     "x": -500.0,  # ft
#     "y": 100.0,
#     "z": 1000.0,
#     "u": 10.0,  # ft/s
#     "v": 0.1,
#     "w": 5.0,
#     "phi": 10 * DEG_TO_RAD,  # deg -> rad
#     "theta": 2 * DEG_TO_RAD,
#     "psi": 180.0 * DEG_TO_RAD,
#     "p": 0.0 * DEG_TO_RAD,  # deg/s -> rad/s
#     "q": 0.0 * DEG_TO_RAD,
#     "r": 0.0 * DEG_TO_RAD,
# }
# heading_controller = controllers.TwelveStateHeadingController(
#         kp=1,
#         ki=0.0,
#         kd=0.1,
#         max_output=2.0,
#         min_output=-2.0,
#         rate_limit=0.5,
#         spiral_mode=True,
#         inner_spiral_range=10.0,
#         outer_spiral_range=100.0,
#     )


### Build Dynamics, Control Sequence, and Initial State (Jann)

In [3]:
jann_continuous_dynamics = JannParafoil3DOF(jann_3dof_params)
jann_discrete_dynamics = get_discrete_time_dynamics(jann_continuous_dynamics, dt)

# # control sequence:
# us_jann = jnp.array(
#     [
#         jnp.ones(N) * 0.1,  # delta_a
#         jnp.zeros(N),  # delta_s
#     ]
# ).T
# # columns: delta_a, delta_s
# # shape (N, m) aka (time steps, control dim)

# build a control input that starts at zero, ramps up to ramp_max
# from T/2 to (T/2 + ramp_time), and stays at ramp_max indefinitely
ramp_max = 0.5  # asymetric deflection
ramp_time = 3.0  # seconds
ramp_start_time = ts[N // 2]
u_interp = jnp.array([0, 0, ramp_max, ramp_max])
t_interp = jnp.array([0, ramp_start_time, ramp_start_time + ramp_time, ts[-1]])
us_jann = jnp.interp(ts, t_interp, u_interp)

# wind disturbance
wind_x = jnp.zeros(N)
wind_y = jnp.ones(N) * -5  # ft/s
wind_z = jnp.zeros(N)
ds_jann = jnp.stack(
    [wind_x, wind_y, wind_z], axis=1
)  # shape (N, 3) for jax.lax.scan
# OR
ds_jann = jnp.zeros((N, 3))  # no wind disturbance

# initial state:
x0_jann = jnp.array(
    [3.0, 3.0, 10 * DEG_TO_RAD, 0.0]
)  # u (m/s), w (m/s) phi (rad), psi (rad)

# trying 12-state version
jann_initial_state = {
    "x_ned": -300.0,  # m
    "y_ned": 200.0,
    "z_ned": -500.0,
    "u": 10.0,  # m/s
    "v": 0.0,  # no sideslip
    "w": 5.0,
    "phi": 10 * DEG_TO_RAD,  # deg -> rad
    "theta": 0 * DEG_TO_RAD,
    "psi": 180.0 * DEG_TO_RAD,
    "p": 0.0 * DEG_TO_RAD,  # deg/s -> rad/s
    "q": 0.0 * DEG_TO_RAD,
    "r": 0.0 * DEG_TO_RAD,
}

jann_3dof_initial_state = {
    "x_ned": -300.0,  # m
    "y_ned": 200.0,
    "z_ned": -500.0,
    "u": jann_3dof_params["u_0"],  # m/s
    "v": 0.0,  # no sideslip
    "w": jann_3dof_params["w_0"],
    "phi": 0.0 * DEG_TO_RAD,  # deg -> rad
    "theta": 0.0 * DEG_TO_RAD,
    "psi": 120.0 * DEG_TO_RAD,
    "p": 0.0 * DEG_TO_RAD,  # deg/s -> rad/s
    "q": 0.0 * DEG_TO_RAD,
    "r": 0.0 * DEG_TO_RAD,
}


x0_jann = jnp.array(list(jann_3dof_initial_state.values()))

### Build Dynamics, Control Sequence, and Initial State (Slegers)

In [4]:
slegers_continuous_dynamics = SlegersParafoil6DOF(slegers_6dof_nonlinear_params)
slegers_discrete_dynamics = get_discrete_time_dynamics(slegers_continuous_dynamics, dt)

# control sequence:

# us_slegers = jnp.ones(N) * 1  # delta_a
us_slegers = jnp.zeros(N)  # no control
us_slegers = us_slegers.at[N // 2 :].set(2.0)  # second half turn

# build a control input that starts at zero, ramps up to ramp_max
# from T/2 to (T/2 + ramp_time), and stays at ramp_max indefinitely
ramp_max = 0.5  # asymetric deflection (rad)
ramp_time = 3.0  # seconds
ramp_start_time = ts[N // 2]
u_interp = jnp.array([0, 0, ramp_max, ramp_max])
t_interp = jnp.array([0, ramp_start_time, ramp_start_time + ramp_time, ts[-1]])
# us_slegers = jnp.interp(ts, t_interp, u_interp)

# wind disturbance
wind_x = jnp.zeros(N)
wind_y = jnp.ones(N) * -5  # ft/s
wind_z = jnp.zeros(N)
ds_slegers = jnp.stack(
    [wind_x, wind_y, wind_z], axis=1
)  # shape (N, 3) for jax.lax.scan
# OR
ds_slegers = jnp.zeros((N, 3))  # no wind disturbance

# Initial state (imperial units):
slegers_initial_state = {
    "x_ned": -300.0,  # ft
    "y_ned": 200.0,
    "z_ned": -500.0,
    "u": 10.0,  # ft/s
    "v": 0.1,
    "w": 5.0,
    "phi": 10 * DEG_TO_RAD,  # deg -> rad
    "theta": 2 * DEG_TO_RAD,
    "psi": 180.0 * DEG_TO_RAD,
    "p": 0.0 * DEG_TO_RAD,  # deg/s -> rad/s
    "q": 0.0 * DEG_TO_RAD,
    "r": 0.0 * DEG_TO_RAD,
}

x0_slegers = jnp.array(list(slegers_initial_state.values()))

### Simulate

Jann

In [5]:
start_time = time.time()
# xs = simulate(x0_jann, us_jann, ds_jann, ts, jann_discrete_dynamics)
# xs = simulate(x0_slegers, us_slegers, ds_slegers, ts, slegers_discrete_dynamics)
# us = us_slegers  # for plotting
# us = us_jann  # for plotting
ctrl_state0 = (
    jnp.array(0.0),  # integral
    jnp.array(0.0),  # prev_error
    jnp.array(0.0),  # prev_control_input
    jnp.array(False),  # spiraling flag
)
heading_controller = controllers.TwelveStateHeadingController(
    kp=1,
    ki=0.0,
    kd=0.1,
    max_output=1.0,
    min_output=-1.0,
    rate_limit=0.2,
    spiral_mode=True,
    inner_spiral_range=10.0,
    outer_spiral_range=100.0,
)
# dummy_controller = controllers.DummyController()
# dummy_controller_state0 = None # dummy controller has no state
xs, us = simulate_with_controller(
    x0_jann, jann_discrete_dynamics, heading_controller, ctrl_state0, ds_jann, ts
)
end_time = time.time()
print(f"Simulation run time: {end_time - start_time:.4f} seconds")
print(f"Total simulation frames: {N}")

Simulation run time: 0.3501 seconds
Total simulation frames: 10000


Slegers

In [6]:
start_time = time.time()
# xs = simulate(x0_slegers, us_slegers, ds_slegers, ts, slegers_discrete_dynamics)
# us = us_slegers  # for plotting
ctrl_state0 = (
    jnp.array(0.0),  # integral
    jnp.array(0.0),  # prev_error
    jnp.array(0.0),  # prev_control_input
    jnp.array(False),  # spiraling flag
)
heading_controller = controllers.TwelveStateHeadingController(
    kp=1,
    ki=0.0,
    kd=0.1,
    max_output=3.0,
    min_output=-3.0,
    rate_limit=0.5,
    spiral_mode=True,
    inner_spiral_range=10.0,
    outer_spiral_range=100.0,
)
# dummy_controller = controllers.DummyController()
# dummy_controller_state0 = None # dummy controller has no state
# xs, us = simulate_with_controller(
#     x0_slegers,
#     slegers_discrete_dynamics,
#     heading_controller,
#     ctrl_state0,
#     ds_slegers,
#     ts,
# )
end_time = time.time()
print(f"Simulation run time: {end_time - start_time:.4f} seconds")
print(f"(total simulation frames: {N})")

Simulation run time: 0.0010 seconds
(total simulation frames: 10000)


### Plot Trajectory

In [7]:
label_idx_to_plot = {
    # "x (ft)": 0,
    # "y (ft)": 1,
    "z (ft)": 2,
    "u (ft/s)": 3,
    "v (ft/s)": 4,
    "w (ft/s)": 5,
    # "phi (deg)": 6,
    # "theta (deg)": 7,
    "psi (deg)": 8,
    # "p (deg/s)": 9,
    # "q (deg/s)": 10,
    "r (deg/s)": 11,
}
labels = list(label_idx_to_plot.keys())
indices = list(label_idx_to_plot.values())

plot_3D_traj(xs, ts)
plot_states(xs, labels, indices, us, ts)

# plot_jann_body(xs, ts)